# Transcoders: Bridging Nonlinear Layers

**Survey Reference:** Section 6.5.1 - Transcoders: Bridging Nonlinear Layers


Standard Sparse Autoencoders (SAEs) excel at decomposing representations in the residual stream, where features propagate linearly. However, they struggle with **nonlinear transformations** like MLP layers and attention mechanisms.

Consider a transformer MLP:

$$\text{MLP}(x) = W_{out} \cdot \sigma(W_{in} \cdot x + b_{in}) + b_{out}$$

We can train SAEs on:
- **MLP inputs** $x$: Features present *before* transformation
- **MLP outputs** $\text{MLP}(x)$: Features present *after* transformation

But neither captures the **causal relationship**: "This input feature causes this output feature." The nonlinearity $\sigma$ distorts the feature space, making it difficult to track how concepts evolve through the transformation.

This notebook explores Transcoders and cross-layer transcoders (CLTs):

1. **Transcoders**
2. **Cross-Layer Transcoders (CLTs)**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

## Transcoder Architecture

A transcoder maps from MLP input to MLP output:

$$f = \text{ReLU}(W_e \cdot x + b_e)$$
$$\hat{y} = W_d \cdot f + b_d$$

Where:
- $x$ is the MLP input (residual stream)
- $y$ is the MLP output  
- $f$ is the sparse feature representation
- The transcoder learns to approximate $y \approx \hat{y}$

**Key difference from SAE**: Transcoders predict the output, not reconstruct the input.

In [ ]:
class Transcoder(nn.Module):
    """
    Transcoder: Maps MLP input -> sparse features -> MLP output.
    
    Unlike SAEs, transcoders model the transformation, not just the representation.
    """
    
    def __init__(self, d_model: int, n_features: int):
        super().__init__()
        self.d_model = d_model
        self.n_features = n_features
        
        # Encoder: map input to sparse features
        self.W_enc = nn.Parameter(torch.randn(d_model, n_features) / np.sqrt(d_model))
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        
        # Decoder: map features to output space
        self.W_dec = nn.Parameter(torch.randn(n_features, d_model) / np.sqrt(n_features))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
    
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode MLP input to sparse features."""
        pre_acts = x @ self.W_enc + self.b_enc
        return F.relu(pre_acts)
    
    def decode(self, f: torch.Tensor) -> torch.Tensor:
        """Decode features to MLP output space."""
        return f @ self.W_dec + self.b_dec
    
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass.
        
        Args:
            x: MLP input [batch, d_model]
            
        Returns:
            y_hat: Predicted MLP output [batch, d_model]
            f: Sparse feature activations [batch, n_features]
        """
        f = self.encode(x)
        y_hat = self.decode(f)
        return y_hat, f
    
    def get_feature_effect(self, feature_idx: int) -> torch.Tensor:
        """
        Get the effect of a feature on the output.
        
        Returns the decoder column for this feature (what it adds to output).
        """
        return self.W_dec[feature_idx]

The transcoder loss predicts the MLP output (not reconstruction of input):

$$\mathcal{L} = \|y - \hat{y}\|_2^2 + \lambda \|f\|_1$$

This encourages:
- **Accurate prediction** of MLP outputs
- **Sparse features** that capture essential computations

In [ ]:
def transcoder_loss(y_true: torch.Tensor, y_pred: torch.Tensor, 
                    f: torch.Tensor, l1_coef: float = 1e-3) -> tuple[torch.Tensor, dict]:
    """
    Compute transcoder loss.
    
    Args:
        y_true: Actual MLP output
        y_pred: Predicted MLP output from transcoder
        f: Feature activations
        l1_coef: Sparsity coefficient
    """
    # Prediction loss
    pred_loss = (y_true - y_pred).pow(2).mean()
    
    # Sparsity loss
    l1_loss = f.abs().mean()
    
    total_loss = pred_loss + l1_coef * l1_loss
    
    with torch.no_grad():
        # Fraction of variance explained
        fve = 1 - (y_true - y_pred).var() / y_true.var()
        # L0 (average active features)
        l0 = (f > 0).float().sum(dim=1).mean()
    
    metrics = {
        'loss': total_loss.item(),
        'pred_loss': pred_loss.item(),
        'l1_loss': l1_loss.item(),
        'fve': fve.item(),
        'l0': l0.item(),
    }
    
    return total_loss, metrics

## Cross-Layer Transcoders (CLTs)

While transcoders bridge a single layer, **Cross-Layer Transcoders** extend this to span *all* layers, providing a unified sparse decomposition of the entire model.

Each CLT feature $a_\ell$ at layer $\ell$:

1. **Reads** from the residual stream at layer $\ell$:
   $$a_\ell = \text{JumpReLU}(W^\ell_{in}h_\ell)$$

2. **Writes** to MLP outputs of *all subsequent layers* via layer-specific decoders:
   $$\hat{y}_{\ell'} = \sum_{\ell=1}^{\ell'} W^{\ell \rightarrow \ell'}_{out} a_\ell$$

This means:
- Features at layer 1 can directly affect outputs at layers 2, 3, ..., L
- Features at layer 2 can affect layers 3, 4, ..., L
- And so on...

### Training

CLTs are trained jointly across all layers:

$$\mathcal{L} = \sum_{\ell=1}^L \|\hat{y}_\ell - y_\ell\|_2^2 + \lambda \sum_{\ell=1}^L \sum_i \tanh(c \cdot \|W^\ell_{out,i}\| \cdot a^\ell_i)$$

where $c$ controls the sharpness of the sparsity penalty.

### Replacement Models

CLTs enable **replacement models**: versions where all MLP layers are replaced by CLT reconstructions.

**Running a replacement model:**
1. Compute CLT feature activations $a_\ell$ at each layer
2. Replace MLP outputs $y_\ell$ with reconstructions $\hat{y}_\ell$
3. Keep attention layers unchanged

In [ ]:
class CrossLayerTranscoder(nn.Module):
    """
    Cross-Layer Transcoder: Features span all layers.
    
    Each feature at layer l can write to all subsequent layers l' > l.
    """
    
    def __init__(self, d_model: int, n_features: int, n_layers: int):
        super().__init__()
        self.d_model = d_model
        self.n_features = n_features
        self.n_layers = n_layers
        
        # Encoder for each layer: read from residual stream
        self.encoders = nn.ModuleList([
            nn.Linear(d_model, n_features, bias=True) 
            for _ in range(n_layers)
        ])
        
        # Decoders: feature at layer l writes to all layers l' >= l
        # Shape: [n_layers, n_layers, n_features, d_model]
        # decoders[l][l'] maps features from layer l to output at layer l'
        self.decoders = nn.ParameterList([
            nn.ParameterList([
                nn.Parameter(torch.randn(n_features, d_model) / np.sqrt(n_features))
                if layer_out >= layer_in else None
                for layer_out in range(n_layers)
            ])
            for layer_in in range(n_layers)
        ])
    
    def encode_layer(self, h: torch.Tensor, layer_idx: int) -> torch.Tensor:
        """Encode residual stream at layer to features using JumpReLU (approximated as ReLU)."""
        return F.relu(self.encoders[layer_idx](h))
    
    def decode_to_layer(self, features_dict: dict, target_layer: int) -> torch.Tensor:
        """
        Decode features from all previous layers to target layer output.
        
        Args:
            features_dict: {layer_idx: features} for all layers <= target_layer
            target_layer: Which layer output to reconstruct
            
        Returns:
            Reconstructed MLP output for target_layer
        """
        output = 0
        for layer_in in range(target_layer + 1):
            if layer_in in features_dict and self.decoders[layer_in][target_layer] is not None:
                # Features from layer_in contribute to target_layer output
                output = output + features_dict[layer_in] @ self.decoders[layer_in][target_layer]
        return output
    
    def forward(self, residual_streams: list[torch.Tensor]) -> tuple[list[torch.Tensor], dict]:
        """
        Forward pass through all layers.
        
        Args:
            residual_streams: List of [batch, d_model] tensors for each layer
            
        Returns:
            reconstructions: List of reconstructed MLP outputs for each layer
            features_dict: Dictionary mapping layer_idx -> feature activations
        """
        features_dict = {}
        reconstructions = []
        
        for layer_idx, h in enumerate(residual_streams):
            # Encode current layer
            features_dict[layer_idx] = self.encode_layer(h, layer_idx)
            
            # Decode to current layer output using all features up to this layer
            recon = self.decode_to_layer(features_dict, layer_idx)
            reconstructions.append(recon)
        
        return reconstructions, features_dict